# Baseline analysis — `baseline_unet3d`

Everything the write-up needs for the 3D U-Net baseline: training curves from the W&B API, the
per-region results table in Markdown and LaTeX, per-case Dice distributions, and a qualitative
best/median/worst figure.

**Inputs** (all produced by earlier stages, nothing is recomputed here):

| Source | Produced by |
|---|---|
| W&B run `nz5y7li7` | the two training sessions, synced from offline |
| `outputs/eval_test/per_case_metrics.csv` | `scripts/evaluate.py` on the **test** split |
| `outputs/eval_test/predictions/*.npy` | same run — uint8 class maps in ORIGINAL geometry |
| `data/preprocessed/brats/<case>/` | `scripts/preprocess.py` — image/label in CROPPED geometry |

The geometry mismatch in the last two rows is the one real trap in this notebook and is handled
explicitly in the qualitative section.

Figures are written to `outputs/figures/`.

In [ ]:
# --- Config: the only cell to edit ---
# NOTE: the W&B entity is NOT the username. `wandb.entity: null` resolves to the
# default entity, which for this account is an institutional one.
WANDB_ENTITY = "amishyadav126-svkm-s-narsee-monjee-institute-of-manageme"
WANDB_PROJECT = "neurovision-x"
WANDB_RUN_ID = "nz5y7li7"

EVAL_DIR = "../outputs/eval_test"
PREP_DIR = "../data/preprocessed/brats"
FIG_DIR = "../outputs/figures"

MODEL_LABEL = "3D U-Net (baseline)"
# Display order. BraTS papers conventionally report WT, TC, ET -- outermost to
# innermost -- which is the reverse of the (ET, TC, WT) channel order the model
# and metrics use. Getting this backwards silently mislabels every table.
REGIONS = ["WT", "TC", "ET"]
REGION_LONG = {"WT": "Whole tumour", "TC": "Tumour core", "ET": "Enhancing tumour"}

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

EVAL_DIR, PREP_DIR, FIG_DIR = Path(EVAL_DIR), Path(PREP_DIR), Path(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)

per_case = pd.read_csv(EVAL_DIR / "per_case_metrics.csv", index_col=0)
summary = pd.read_csv(EVAL_DIR / "summary.csv", index_col=0)
print(f"{len(per_case)} test cases loaded from {EVAL_DIR}")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
REGION_COLOR = {"WT": "#2a9d8f", "TC": "#e76f51", "ET": "#264653"}

## 1. Training curves from the W&B API

The run is a single continuous history across both Kaggle sessions — session 2 resumed into the
same run via the id stored in the checkpoint, so there is no seam to stitch here.

In [ ]:
import wandb

api = wandb.Api()
run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_RUN_ID}")
print(f"{run.name} ({run.id})  state={run.state}")

# samples large enough to return the full history rather than a subsample --
# wandb downsamples by default, which would visibly smooth the curves.
hist = run.history(samples=100_000, pandas=True)
print(f"history rows: {len(hist)}   columns: {len(hist.columns)}")


def series(col):
    """Return (step, value) for one logged metric, dropping steps where it is absent.

    Train and validation metrics are logged at different cadences (validation
    every 2 epochs), so the history is sparse and a plain plot would connect
    across NaNs.
    """
    if col not in hist.columns:
        return None, None
    sub = hist[["_step", col]].dropna()
    return sub["_step"].to_numpy(), sub[col].to_numpy()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

ax = axes[0]
for col, lab, c in [("train/loss_epoch", "train loss (epoch mean)", "#264653")]:
    x, y = series(col)
    if x is not None:
        ax.plot(x, y, color=c, lw=1.2, label=lab)
ax.set_xlabel("optimizer step"); ax.set_ylabel("DiceBCE loss")
ax.set_title("Training loss"); ax.legend(frameon=False)

ax = axes[1]
for r in REGIONS:
    x, y = series(f"val/dice_{r}")
    if x is not None:
        ax.plot(x, y, color=REGION_COLOR[r], lw=1.2, label=r)
x, y = series("val/dice_mean")
if x is not None:
    ax.plot(x, y, color="k", lw=1.6, ls="--", label="mean")
ax.set_xlabel("optimizer step"); ax.set_ylabel("Dice")
ax.set_ylim(0, 1); ax.set_title("Validation Dice"); ax.legend(frameon=False, ncol=2)

ax = axes[2]
x, y = series("train/lr")
if x is not None:
    ax.plot(x, y, color="#e76f51", lw=1.2)
ax.set_xlabel("optimizer step"); ax.set_ylabel("learning rate")
ax.set_title("LR schedule (warmup + cosine)")

fig.suptitle(f"{MODEL_LABEL} — {run.name}", y=1.04)
fig.savefig(FIG_DIR / "training_curves.png")
plt.show()

## 2. Results table — Markdown and LaTeX

Reported as **mean ± std** across the 189 held-out test cases, with the median alongside. The
median matters here: the Dice distributions are strongly left-skewed, so the mean alone
understates typical performance while hiding how bad the failures are.

`n_missing` is the number of cases where HD95 was genuinely undefined — one side empty, the other
not. Those are excluded from the HD95 mean rather than given an arbitrary penalty, so the count
has to be reported with it.

In [ ]:
def build_results_table(pc: pd.DataFrame, regions=REGIONS) -> pd.DataFrame:
    """One row per region: Dice/IoU mean±std and median, HD95 mean±std, and counts."""
    rows = []
    for r in regions:
        d, i, h = pc[f"dice_{r}"], pc[f"iou_{r}"], pc[f"hd95_{r}"]
        rows.append({
            "Region": r,
            "Dice mean": d.mean(), "Dice std": d.std(), "Dice median": d.median(),
            "IoU mean": i.mean(), "IoU std": i.std(),
            "HD95 mean": h.mean(), "HD95 std": h.std(), "HD95 median": h.median(),
            "HD95 n_missing": int(h.isna().sum()),
            "GT empty %": 100.0 * pc[f"gt_empty_{r}"].mean(),
        })
    return pd.DataFrame(rows).set_index("Region")


table = build_results_table(per_case)
display(table.round(4))

In [ ]:
def to_markdown(tbl: pd.DataFrame, n_cases: int, label: str) -> str:
    lines = [
        f"**{label}** — BraTS 2021, {n_cases} held-out test cases.",
        "",
        "| Region | Dice (mean ± std) | Dice median | IoU (mean ± std) | HD95 mm (mean ± std) | HD95 median | HD95 undefined | GT empty |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for r, row in tbl.iterrows():
        lines.append(
            f"| {r} | {row['Dice mean']:.4f} ± {row['Dice std']:.4f} | {row['Dice median']:.4f} "
            f"| {row['IoU mean']:.4f} ± {row['IoU std']:.4f} "
            f"| {row['HD95 mean']:.2f} ± {row['HD95 std']:.2f} | {row['HD95 median']:.2f} "
            f"| {int(row['HD95 n_missing'])}/{n_cases} | {row['GT empty %']:.1f}% |"
        )
    return "\n".join(lines)


def to_latex(tbl: pd.DataFrame, n_cases: int, label: str) -> str:
    """booktabs table. Requires \\usepackage{booktabs} in the document preamble."""
    out = [
        r"\begin{table}[t]", r"\centering",
        rf"\caption{{{label} on BraTS 2021 ({n_cases} held-out test cases). "
        r"Dice and IoU use the \texttt{ignore\_empty=False} convention; HD95 excludes cases where "
        r"it is undefined (one side empty), counted in the last column.}",
        r"\label{tab:baseline}",
        r"\begin{tabular}{lccccc}", r"\toprule",
        r"Region & Dice & Dice (med.) & IoU & HD95 (mm) & HD95 undef. \\",
        r"\midrule",
    ]
    for r, row in tbl.iterrows():
        out.append(
            rf"{r} & ${row['Dice mean']:.3f} \pm {row['Dice std']:.3f}$ & ${row['Dice median']:.3f}$ "
            rf"& ${row['IoU mean']:.3f} \pm {row['IoU std']:.3f}$ "
            rf"& ${row['HD95 mean']:.2f} \pm {row['HD95 std']:.2f}$ "
            rf"& {int(row['HD95 n_missing'])}/{n_cases} \\"
        )
    out += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(out)


md = to_markdown(table, len(per_case), MODEL_LABEL)
tex = to_latex(table, len(per_case), MODEL_LABEL)
(FIG_DIR / "results_table.md").write_text(md + "\n")
(FIG_DIR / "results_table.tex").write_text(tex + "\n")
print(md)
print("\n" + "=" * 70 + "\n")
print(tex)

## 3. Per-case Dice distributions

The point of this figure is the tail, not the box. A mean of 0.90 with a median of 0.95 means the
typical case is much better than the average — and that a minority of cases fail badly enough to
move the mean several points. For a claim about *reliability*, those are the cases that matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
data = [per_case[f"dice_{r}"].dropna().to_numpy() for r in REGIONS]
# tick_labels, not labels: matplotlib removed the `labels` kwarg.
bp = ax.boxplot(data, tick_labels=REGIONS, showfliers=False, widths=0.55, patch_artist=True,
                medianprops=dict(color="black", lw=1.6))
for patch, r in zip(bp["boxes"], REGIONS):
    patch.set_facecolor(REGION_COLOR[r]); patch.set_alpha(0.35)
# Every case drawn individually: the outliers ARE the story, so hiding them
# behind showfliers=False and overlaying jitter shows density and tail at once.
rng = np.random.default_rng(0)
for i, (r, vals) in enumerate(zip(REGIONS, data), start=1):
    ax.scatter(rng.normal(i, 0.055, len(vals)), vals, s=7, alpha=0.35,
               color=REGION_COLOR[r], edgecolors="none", zorder=3)
ax.set_ylabel("Dice"); ax.set_ylim(-0.02, 1.02)
ax.set_title(f"Per-case Dice, {len(per_case)} test cases")

ax = axes[1]
for r in REGIONS:
    v = np.sort(per_case[f"dice_{r}"].dropna().to_numpy())
    ax.plot(v, np.linspace(0, 100, len(v)), color=REGION_COLOR[r], lw=1.6, label=r)
ax.set_xlabel("Dice"); ax.set_ylabel("% of cases at or below")
ax.set_xlim(0, 1); ax.set_title("Cumulative distribution"); ax.legend(frameon=False)
# 0.80 is a common "clinically usable" rule of thumb; the gap between regions at
# this line is a more honest reliability summary than any single mean.
ax.axvline(0.8, color="grey", ls=":", lw=1)

fig.savefig(FIG_DIR / "per_case_dice.png")
plt.show()

print("fraction of cases below Dice 0.80:")
for r in REGIONS:
    v = per_case[f"dice_{r}"]
    print(f"  {r}: {100.0 * (v < 0.8).mean():5.1f}%   (worst case {v.min():.4f})")

## 4. Qualitative — best / median / worst

**The geometry trap.** Saved predictions are in ORIGINAL BraTS geometry (240×240×155), because
that is what a submission requires. The preprocessed image and label are CROPPED to the case's
nonzero bounding box. Overlaying them directly would silently misalign by the crop offset, and
the result would look plausible. Each prediction is therefore cropped back with the same `bbox`
from `meta.json` that `uncrop_to_original` used, and the shapes are asserted to match.

In [ ]:
def load_case(case_id: str):
    """Load image, ground-truth label and prediction, all in CROPPED geometry.

    Returns:
        `(image, gt, pred)` with shapes `(4, D, H, W)`, `(D, H, W)`, `(D, H, W)`.

    Raises:
        ValueError: If the prediction's bbox-cropped shape does not match the
            stored image, which would mean prediction and meta.json came from
            different preprocessing runs.
    """
    case_dir = PREP_DIR / case_id
    meta = json.loads((case_dir / "meta.json").read_text())
    image = np.load(case_dir / "image.npy").astype(np.float32)
    gt = np.load(case_dir / "label.npy")

    pred_full = np.load(EVAL_DIR / "predictions" / f"{case_id}.npy")
    if tuple(pred_full.shape) != tuple(meta["original_shape"]):
        raise ValueError(f"{case_id}: prediction {pred_full.shape} != original {meta['original_shape']}")
    sl = tuple(slice(int(a), int(b)) for a, b in meta["bbox"])
    pred = pred_full[sl]
    if pred.shape != gt.shape:
        raise ValueError(f"{case_id}: cropped prediction {pred.shape} != label {gt.shape}")
    return image, gt, pred


def best_axial_slice(gt: np.ndarray) -> int:
    """Index of the axial slice with the most ground-truth tumour voxels."""
    return int(np.argmax((gt > 0).sum(axis=(0, 1))))


def to_wt(label: np.ndarray) -> np.ndarray:
    """Whole-tumour binary mask from a `{0,1,2,3}` class map."""
    return label > 0


# Class overlay colours, matching src/neurovision/visualization/qc.py:
# 1 = NCR/NET, 2 = ED, 3 = ET. Class 0 is transparent.
CLASS_CMAP = ListedColormap([(0, 0, 0, 0), "#56b4e9", "#009e73", "#d55e00"])
print("helpers ready")

In [ ]:
ranked = per_case["dice_mean"].sort_values()
picks = [
    ("worst", ranked.index[0], ranked.iloc[0]),
    ("median", ranked.index[len(ranked) // 2], ranked.iloc[len(ranked) // 2]),
    ("best", ranked.index[-1], ranked.iloc[-1]),
]
for tag, cid, val in picks:
    print(f"{tag:7s} {cid}  dice_mean={val:.4f}  "
          f"(ET {per_case.loc[cid,'dice_ET']:.3f} TC {per_case.loc[cid,'dice_TC']:.3f} "
          f"WT {per_case.loc[cid,'dice_WT']:.3f})")

fig, axes = plt.subplots(len(picks), 4, figsize=(12.5, 3.05 * len(picks)))
for row, (tag, cid, val) in enumerate(picks):
    image, gt, pred = load_case(cid)
    z = best_axial_slice(gt)
    t1ce = image[1, :, :, z]   # channel order is [t1, t1ce, t2, flair]
    flair = image[3, :, :, z]
    g, p = gt[:, :, z], pred[:, :, z]

    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    axes[row][0].imshow(flair.T, cmap="gray", origin="lower")
    axes[row][0].set_ylabel(f"{tag}\n{cid}\ndice={val:.3f}", fontsize=8)
    axes[row][0].set_title("FLAIR" if row == 0 else "")

    axes[row][1].imshow(t1ce.T, cmap="gray", origin="lower")
    axes[row][1].imshow(np.ma.masked_where(g == 0, g).T, cmap=CLASS_CMAP, vmin=0, vmax=3,
                        alpha=0.55, origin="lower", interpolation="nearest")
    axes[row][1].set_title("Ground truth" if row == 0 else "")

    axes[row][2].imshow(t1ce.T, cmap="gray", origin="lower")
    axes[row][2].imshow(np.ma.masked_where(p == 0, p).T, cmap=CLASS_CMAP, vmin=0, vmax=3,
                        alpha=0.55, origin="lower", interpolation="nearest")
    axes[row][2].set_title("Prediction" if row == 0 else "")

    # Agreement on whole tumour: where the model over- and under-segments is more
    # diagnostic than either mask alone.
    gw, pw = to_wt(g), to_wt(p)
    agree = np.zeros(g.shape, dtype=np.uint8)
    agree[gw & pw] = 1     # true positive
    agree[~gw & pw] = 2    # false positive
    agree[gw & ~pw] = 3    # false negative
    axes[row][3].imshow(flair.T, cmap="gray", origin="lower")
    axes[row][3].imshow(np.ma.masked_where(agree == 0, agree).T,
                        cmap=ListedColormap([(0, 0, 0, 0), "#4daf4a", "#e41a1c", "#377eb8"]),
                        vmin=0, vmax=3, alpha=0.6, origin="lower", interpolation="nearest")
    axes[row][3].set_title("WT agreement" if row == 0 else "")

handles = [
    plt.Line2D([], [], marker="s", ls="", color="#56b4e9", label="NCR/NET"),
    plt.Line2D([], [], marker="s", ls="", color="#009e73", label="ED"),
    plt.Line2D([], [], marker="s", ls="", color="#d55e00", label="ET"),
    plt.Line2D([], [], marker="s", ls="", color="#4daf4a", label="TP (WT)"),
    plt.Line2D([], [], marker="s", ls="", color="#e41a1c", label="FP (WT)"),
    plt.Line2D([], [], marker="s", ls="", color="#377eb8", label="FN (WT)"),
]
fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"{MODEL_LABEL} — qualitative results (axial slice with most tumour)", y=1.0)
fig.savefig(FIG_DIR / "qualitative_best_median_worst.png")
plt.show()

## 5. What the worst case actually is

Worth naming explicitly rather than leaving it as a low number in a table — the failure mode is
the input to the fusion design.

In [ ]:
worst = per_case["dice_mean"].idxmin()
row = per_case.loc[worst]
print(f"worst case: {worst}")
for r in REGIONS:
    print(f"  {r}: dice={row[f'dice_{r}']:.4f}  hd95={row[f'hd95_{r}']:.2f} mm  "
          f"gt_empty={bool(row[f'gt_empty_{r}'])}")

print("\ncases with a completely failed region (Dice = 0):")
for r in REGIONS:
    zeros = per_case.index[per_case[f"dice_{r}"] == 0].tolist()
    print(f"  {r}: {len(zeros)} case(s) {zeros[:5]}")

print("\nfiles written:")
for f in sorted(FIG_DIR.iterdir()):
    print("  ", f)